<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/bonus/notebooks/11_topic_models_with_bertopic_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Bonus: Topic Models with BERTopic

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

A topic model finds the themes in a collection of documents without being told what to look for.
The classic method, LDA, works on bags of words. **BERTopic** (Grootendorst, 2022) builds topics from
sentence embeddings instead: it groups documents that *mean* similar things, then describes each group
with its most characteristic words.

Thursday's session on sentence transformers and document clustering introduced the building blocks.
This notebook puts them together in BERTopic, and spends most of its time on the questions that decide
whether you can trust a topic model: how much do the results depend on your choices and on chance, and
do the topics match anything we know?

In this notebook you will:

- go through BERTopic's four steps: embed, reduce, cluster, describe
- fit a model on US Congressional bill titles and inspect its topics
- validate the topics against the bills' known policy areas
- see how one setting changes the number of topics, and how much topics vary between runs
- deal with outliers, track topics over time, and compare with LDA

**Contents**

1. How BERTopic works
2. The data
3. Embeddings, computed once
4. A first model
5. Looking at topics
6. Validation: do topics match anything we know?
7. Your choices shape the topics
8. Outliers
9. Stability: how much is chance?
10. Topics over time
11. Comparing with LDA
12. Better labels, and good practice

A GPU makes the embedding step faster, but a CPU works fine for this data (about a minute).

There are 5 exercises along the way, marked ✏️. A separate solutions notebook has worked answers, but try them yourself first.

## 0. Setup

In [ ]:
!pip install -q bertopic

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from bertopic import BERTopic
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from umap import UMAP

SEED = 42
device = "cuda" if torch.cuda.is_available() else "cpu"
pd.set_option("display.max_colwidth", 110)
print("Running on", device.upper())

## 1. How BERTopic works

BERTopic is a pipeline of four steps, each done by a separate, replaceable tool:

| Step | Default tool | What it does |
|---|---|---|
| 1. **Embed** | a sentence transformer | turns each document into a vector that captures its meaning |
| 2. **Reduce** | UMAP | squeezes the 384 dimensions into about 5, keeping nearby documents nearby |
| 3. **Cluster** | HDBSCAN | finds dense groups of documents; documents in no group become **outliers** (topic `-1`) |
| 4. **Describe** | c-TF-IDF | finds the words that are typical for each cluster compared to the others |

Step 4 is TF-IDF (Day 1) with a twist: all documents in a cluster are glued together into one big
"class document", and words are weighted by how frequent they are in that class compared to all
classes. The top words become the topic's description.

Two differences from LDA matter for interpretation:

- Each document belongs to **one** topic (or none), while LDA gives each document a *mix* of topics.
- The number of topics isn't set in advance: HDBSCAN decides, driven by `min_cluster_size`.

## 2. The data

The same source as the Day 3 BERT notebook and Bonus 6-8: titles of US Congressional bills. This time
we take **eight** policy areas, 300 bills each, plus the Congress in which each bill was introduced.
The topic model never sees the policy areas; we keep them to check its topics afterwards.

In [ ]:
from huggingface_hub import hf_hub_download

parts = []
for c in [0, 5]:
    path = hf_hub_download(repo_id="dreamproit/bill_labels_us", repo_type="dataset",
                           revision="refs/convert/parquet", filename=f"default/train/{c:04d}.parquet")
    parts.append(pd.read_parquet(path, columns=["title", "policy_area", "congress"]))
raw = pd.concat(parts, ignore_index=True)

AREAS = ["Health", "Education", "Taxation", "Armed Forces and National Security",
         "Environmental Protection", "Agriculture and Food", "Immigration",
         "Government Operations and Politics"]
bills = (raw.assign(title=raw["title"].fillna("").str.strip())
            .query("title != ''")
            .drop_duplicates(subset="title")
            .query("policy_area in @AREAS")
            .sample(frac=1, random_state=SEED)
            .groupby("policy_area").head(300)
            .reset_index(drop=True))
docs = bills["title"].tolist()

print(len(bills), "bills")
print(bills["policy_area"].value_counts().to_string())
bills.head()

## 3. Embeddings, computed once

The embedding step is the slow part, and it's the same however we set up the rest. So we compute the
embeddings once, save them, and pass them to every model we fit. `all-MiniLM-L6-v2` is small, fast and
good; for Dutch or mixed-language data, use a multilingual model such as
`paraphrase-multilingual-MiniLM-L12-v2`.

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=device)

if os.path.exists("bill_embeddings.npy"):
    embeddings = np.load("bill_embeddings.npy")
else:
    embeddings = embedder.encode(docs, show_progress_bar=True, batch_size=64)
    np.save("bill_embeddings.npy", embeddings)
print(embeddings.shape)

(If you change the data, delete `bill_embeddings.npy` first, or the old embeddings are loaded.)

## 4. A first model

We set up every step explicitly instead of relying on the defaults, so that we know (and can report)
what the model does:

- **UMAP**: `n_neighbors=15` (how local the structure is), `n_components=5`, `min_dist=0.0` (pack
  similar documents tightly, good for clustering), `metric="cosine"`, and a `random_state` so we get
  the same result every time (Section 9 shows why that matters).
- **HDBSCAN**: `min_cluster_size=20`, the smallest group we'd call a topic.
- **CountVectorizer** for the topic descriptions: English stop words *plus* words that appear in almost
  every bill title ("amend", "act", "provide", ...), letters only (no years or section numbers), and
  two-word phrases.

In [ ]:
BILL_WORDS = {"amend", "amends", "act", "acts", "provide", "provides", "purposes", "purpose", "bill",
              "establish", "require", "united", "states", "code", "title", "section", "certain", "law"}
STOP = list(ENGLISH_STOP_WORDS | BILL_WORDS)


def make_model(min_cluster_size=20, seed=SEED, n_neighbors=15):
    return BERTopic(
        embedding_model=embedder,
        umap_model=UMAP(n_neighbors=n_neighbors, n_components=5, min_dist=0.0, metric="cosine", random_state=seed),
        hdbscan_model=HDBSCAN(min_cluster_size=min_cluster_size, metric="euclidean",
                              cluster_selection_method="eom", prediction_data=True),
        vectorizer_model=CountVectorizer(stop_words=STOP, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
                                         ngram_range=(1, 2), min_df=2),
    )


topic_model = make_model()
topics, _ = topic_model.fit_transform(docs, embeddings)
info = topic_model.get_topic_info()
print(f"{len(info) - 1} topics; {np.mean(np.array(topics) == -1):.0%} of bills are outliers (topic -1)")
info.head(12)

`get_topic_info()` lists every topic with its size, an automatic name made of its top words, and a
few representative documents. Topic `-1` collects the outliers: documents HDBSCAN didn't consider part
of any dense group.

## 5. Looking at topics

Top words are a starting point, never the end. **Read documents** from each topic before you name
or interpret it.

In [ ]:
for t in info.Topic[1:5]:
    words = ", ".join(w for w, _ in topic_model.get_topic(t)[:8])
    print(f"Topic {t}: {words}")
    for d in topic_model.get_representative_docs(t)[:3]:
        print("    -", d[:110])
    print()

In [ ]:
topic_model.visualize_barchart(top_n_topics=8, n_words=6)

The interactive plots (made with plotly) are handy for exploring: hover to see the words and scores.

> **✏️ Exercise 1**
>
> Pick three topics. For each, read ten documents (`bills[np.array(topics) == t].title`), not just the
> top words, and write a short label of your own. Give the topics these labels with
> `topic_model.set_topic_labels({topic_id: "label", ...})` and show `get_topic_info()` again. Did
> reading the documents change your idea of any topic?

In [ ]:
# Your code here

## 6. Validation: do topics match anything we know?

Here we have something most topic modelers don't: an independent classification of every document
(the policy area, assigned by the Congressional Research Service). A cross-table shows how topics and
policy areas relate.

In [ ]:
bills["topic"] = topics
crosstab = pd.crosstab(bills["topic"], bills["policy_area"])
crosstab.index = [f"{t}: " + topic_model.get_topic_info().set_index("Topic").loc[t, "Name"][:40]
                  for t in crosstab.index]
crosstab.style.background_gradient(cmap="Greens", axis=1)

A good topic falls mostly in one column. Some policy areas may be split into several topics (one area
can contain several themes), and some topics may mix areas.

Two summary numbers compare the whole partition with the policy areas, from 0 (unrelated) to 1
(identical): **normalized mutual information** (NMI) and the **adjusted Rand index** (ARI). We compute
them without the outliers (and report how many documents that leaves out).

In [ ]:
def agreement_with_areas(topics, labels=bills["policy_area"]):
    topics = np.asarray(topics)
    keep = topics != -1
    return {"topics": len(set(topics[keep])), "outliers": float(np.mean(~keep)),
            "NMI": normalized_mutual_info_score(labels[keep], topics[keep]),
            "ARI": adjusted_rand_score(labels[keep], topics[keep])}

{k: round(v, 3) for k, v in agreement_with_areas(topics).items()}

Keep in mind what this measures. Policy areas are *one* way of dividing the bills, not the "true
topics". A topic model that splits health into "Medicare" and "public health" scores lower on NMI,
but may be more useful. Use the comparison to check that the topics are meaningful at all, and to
understand how they differ from the categories you know.

## 7. Your choices shape the topics

`min_cluster_size` is the most consequential setting: small values give many small topics (and often
fewer outliers per topic but more topics to read), large values give a few broad ones. There's no
correct value; it depends on the level of detail your question needs.

> **✏️ Exercise 2**
>
> Fit models with `min_cluster_size` 10, 20, 40 and 80 (use `make_model(min_cluster_size=...)` and pass
> the same `embeddings`). For each, record the number of topics, the share of outliers, and NMI with
> the policy areas. Put the results in a table. Which value would you choose for (a) a broad overview,
> (b) a detailed map of what Congress works on?

In [ ]:
# Your code here

## 8. Outliers

HDBSCAN often leaves a large share of documents as outliers. That's honest (those documents didn't
fit well anywhere), but it's a problem if you want to count topics in *all* documents.
`reduce_outliers` assigns them to the closest topic, here by comparing their embeddings with the
topics' average embeddings. Afterwards, `update_topics` recomputes the topic descriptions.

> **✏️ Exercise 3**
>
> 1. Use `topic_model.reduce_outliers(docs, topics, strategy="embeddings", embeddings=embeddings)` to get
>    new topic assignments. How many documents moved from `-1` to a topic?
> 2. Read ten of the reassigned titles together with their new topic's top words. Do the assignments
>    make sense?
> 3. Compute NMI for the new assignments (there are no outliers left). Compare it with NMI before.

In [ ]:
# Your code here

## 9. Stability: how much is chance?

UMAP is a random algorithm. With a different `random_state`, the reduced space differs a little, and
so can the clusters. If your topics change substantially between runs, conclusions drawn from one
run are fragile. Always check: fit the same model with several seeds and compare.

> **✏️ Exercise 4**
>
> Fit the model with three different seeds (e.g., 1, 2, 3; same `min_cluster_size`). For each pair of
> runs, compute the adjusted Rand index between their topic assignments (on documents that are not
> outliers in either run). Also compare the number of topics and the top words of the largest topic.
> How stable is the model?

In [ ]:
# Your code here

## 10. Topics over time

`topics_over_time` computes, for each time point, how many documents each topic has and which words
describe it at that time. Here the time points are Congresses (two-year periods).

In [ ]:
tot = topic_model.topics_over_time(docs, bills["congress"].tolist())
topic_model.visualize_topics_over_time(tot, top_n_topics=6)

The counts depend on how many bills per Congress happen to be in our sample, so for real conclusions,
use shares per period rather than counts, and a sample designed for comparisons over time.

In [ ]:
share = pd.crosstab(bills["congress"], bills["topic"], normalize="index")
share.drop(columns=-1, errors="ignore").iloc[:, :6].round(3)

## 11. Comparing with LDA

LDA (latent Dirichlet allocation; Blei et al., 2003) is the classic topic model: each topic is a
distribution over words, each document a mix of topics, estimated from word counts alone. scikit-learn
has an implementation. LDA needs you to choose the number of topics.

> **✏️ Exercise 5**
>
> Fit `LatentDirichletAllocation` with the same number of topics as the BERTopic model, on a
> `CountVectorizer` with the same stop words. Print the top eight words per LDA topic. Then give each
> document its most likely LDA topic (`argmax`) and compute NMI with the policy areas. How does it
> compare with BERTopic? Why might LDA struggle with bill titles in particular?

In [ ]:
# Your code here

## 12. Better labels, and good practice

### Better topic descriptions

c-TF-IDF's top words can be repetitive ("tax", "taxes", "tax credit"). BERTopic can re-rank them:

```python
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
topic_model.update_topics(docs, representation_model=MaximalMarginalRelevance(diversity=0.5))
```

- `KeyBERTInspired` keeps the words whose embeddings are closest to the topic's documents.
- `MaximalMarginalRelevance` trades relevance against variety.
- An LLM (Day 4) can write a short label from the top words and representative documents. Check the
  labels against the documents: a fluent label can still be wrong.

### Good practice

- **Read documents**, not just top words, before naming or interpreting a topic.
- **Report every choice**: embedding model, UMAP and HDBSCAN settings, vectorizer, seed, outlier
  handling. They all change the result.
- **Check stability** across seeds and across reasonable settings; build conclusions on what holds.
- **Validate** against something you know: labels, metadata, a hand-coded sample (Bonus 5), or
  experts reading the topics.
- Topics are a **lens**, not a measurement of "what the documents are really about". For measuring
  a predefined concept, a validated classifier (Day 2-4) is usually the better tool; topic models shine
  for exploring.

### Further reading

- Blei, D. M., Ng, A. Y., & Jordan, M. I. (2003). Latent Dirichlet allocation. *Journal of Machine
  Learning Research, 3*, 993-1022.
- Grootendorst, M. (2022). BERTopic: Neural topic modeling with a class-based TF-IDF procedure.
  *arXiv:2203.05794*.
- McInnes, L., Healy, J., & Melville, J. (2018). UMAP: Uniform manifold approximation and projection
  for dimension reduction. *arXiv:1802.03426*.
- Roberts, M. E., Stewart, B. M., & Tingley, D. (2019). stm: An R package for structural topic models.
  *Journal of Statistical Software, 91*(2), 1-40.